In [0]:
user_logs_path = "/Volumes/workspace/kkbox/raw_data/user_logs_v2.csv"

user_logs_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(user_logs_path)
)

display(user_logs_df.limit(10))

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

user_logs_scoped = (
    user_logs_df
    .join(
        customer_scope.select("msno"),
        on="msno",
        how="inner"
    )
    .select(
        "msno",
        F.to_date(F.col("date").cast("string"), "yyyyMMdd").alias("log_date"),
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    )
)

display(user_logs_scoped.limit(10))

In [0]:
user_logs_scoped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.user_logs_scoped")

In [0]:
user_logs_validation = spark.table("workspace.kkbox.user_logs_scoped").agg(
    F.count("*").alias("total_log_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.min("log_date").alias("earliest_log_date"),
    F.max("log_date").alias("latest_log_date")
)

display(user_logs_validation)

In [0]:
user_logs_quality = spark.table("workspace.kkbox.user_logs_scoped").agg(
    F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
    F.sum(F.when(F.col("log_date").isNull(), 1).otherwise(0)).alias("null_log_dates"),
    F.sum(F.when(F.col("num_25") < 0, 1).otherwise(0)).alias("negative_num_25"),
    F.sum(F.when(F.col("num_50") < 0, 1).otherwise(0)).alias("negative_num_50"),
    F.sum(F.when(F.col("num_75") < 0, 1).otherwise(0)).alias("negative_num_75"),
    F.sum(F.when(F.col("num_985") < 0, 1).otherwise(0)).alias("negative_num_985"),
    F.sum(F.when(F.col("num_100") < 0, 1).otherwise(0)).alias("negative_num_100"),
    F.sum(F.when(F.col("num_unq") < 0, 1).otherwise(0)).alias("negative_num_unq"),
    F.sum(F.when(F.col("total_secs") < 0, 1).otherwise(0)).alias("negative_total_secs")
)

display(user_logs_quality)